<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/cmaes_practical_guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 謝辞


本ノートブックは，日本船舶海洋工学会の主催する第３５４回KFRセミナー「船舶海洋工学者のためのブラックボックス最適化セミナー」にて使用するために取りまとめたものである．以前よりこのような実践ガイドを作成して公開したいと思っていたが，今回のような機会がなかったらいつまでも完成させられずにいたと思う．本セミナーを企画し，今回のような機会を与えてくれた大阪大学の牧敦生先生には深く感謝する．また，第３５４回KFRセミナーにて講演頂いた大阪大学（講演当時博士課程在籍中）の宮内新喜氏からは，CMA-ESの船舶海洋工学分野への応用に関する共同研究や，今回の宮内氏の講演資料を通して，数多くの知見を得ることができた．ここで得られた知見は，応用研究に携わっている研究者であるからこそ得られるものであり，共同研究がなければ得られなかったことと思う．牧先生と宮内氏の両名には深く感謝する．

<!-- EN -->
# Acknowledgment


<!-- EN -->

This notebook was compiled for use at the 354th KFR Seminar, "Black-Box Optimization Seminar for Naval Architects and Ocean Engineers," organized by the Japan Society of Naval Architects and Ocean Engineers. I had long wanted to create and publish such a practical guide, but I suspect it would never have been completed without an opportunity like this one. I am deeply grateful to Professor Atsuo Maki of Osaka University for organizing the seminar and giving me this opportunity. I also received many valuable insights from Mr. Yoshiki Miyauchi (then a doctoral student at Osaka University) who presented at the 354th KFR Seminar, through our joint research on the application of CMA-ES to the field of naval architecture and ocean engineering and through his lecture materials. The insights gained there are ones that could only be obtained by a researcher engaged in applied research, and I do not think they could have been obtained without that collaboration. I am deeply grateful to both Professor Maki and Mr. Miyauchi.


# 目的

勾配を用いない最適化法であるCMA-ESを利用している応用側の研究者から，「CMA-ESを用いてみたが，望ましい解が得られない」といった相談やコメントをしばしばいただきます．ここでは，CMA-ESをうまく活用するための設定，実験結果から問題の性質についての分析，問題定式化の検討の観点から，私の経験則をまとめます．

<!-- EN -->
# Purpose


<!-- EN -->
Applied researchers who use CMA-ES, a gradient-free optimization method, often ask or comment that "we tried CMA-ES but could not obtain a satisfactory solution." Here, I summarize my empirical rules from the perspectives of settings for making effective use of CMA-ES, analysis of problem characteristics from experimental results, and reconsideration of problem formulation.


# 参考資料

この資料は，著者らが毎年国際会議GECCOにおいて開催しているCMA-ESチュートリアルを参考にしています．以下のチュートリアルではCMA-ESの設計原理やアルゴリズムの各コンポーネントの役割を中心に解説しています．一方，本資料では，CMA-ESをうまく活用するための方法に焦点を当てています．

Youhei Akimoto and Nikolaus Hansen. 2022. CMA-ES and advanced adaptation mechanisms. In Proceedings of the Genetic and Evolutionary Computation Conference Companion (GECCO '22). Association for Computing Machinery, New York, NY, USA, 1243–1268. https://doi.org/10.1145/3520304.3533648
チュートリアル動画：https://www.youtube.com/watch?v=7VBKLH3oDuw






<!-- EN -->
# References


<!-- EN -->
This material is based on the CMA-ES tutorial that the authors have held annually at the international conference GECCO. The following tutorial focuses on the design principles of CMA-ES and the role of each component of the algorithm. This material, on the other hand, focuses on methods for making effective use of CMA-ES.

Youhei Akimoto and Nikolaus Hansen. 2022. CMA-ES and advanced adaptation mechanisms. In Proceedings of the Genetic and Evolutionary Computation Conference Companion (GECCO '22). Association for Computing Machinery, New York, NY, USA, 1243–1268. https://doi.org/10.1145/3520304.3533648
Tutorial video: https://www.youtube.com/watch?v=7VBKLH3oDuw


# プログラム

このチュートリアルでは，著者らが公開している最新のCMA-ESの実装である，DD-CMA-ESを用いています．

Y. Akimoto and N. Hansen.
    Diagonal Acceleration for Covariance Matrix Adaptation Evolution Strategies
    Evolutionary Computation (2020) 28(3): 405--435.
コード：https://gist.github.com/youheiakimoto/1180b67b5a0b1265c204cba991fa8518

<!-- EN -->
# Program


<!-- EN -->
In this tutorial, we use DD-CMA-ES, the authors' latest publicly available implementation of CMA-ES.

Y. Akimoto and N. Hansen.
    Diagonal Acceleration for Covariance Matrix Adaptation Evolution Strategies
    Evolutionary Computation (2020) 28(3): 405--435.
Code: https://gist.github.com/youheiakimoto/1180b67b5a0b1265c204cba991fa8518


以下はDD-CMA-ESのアルゴリズムに，矩形制約対処法，周期変数の扱い，リスタート戦略を追加実装したものです．

<!-- EN -->
The following is an implementation that adds box-constraint handling, periodic variable treatment, and a restart strategy to the DD-CMA-ES algorithm.


In [ ]:
!wget https://gist.githubusercontent.com/youheiakimoto/1180b67b5a0b1265c204cba991fa8518/raw/8f3ad2b554680a0c9888904ec2e609e63554a140/ddcma.py
%run ddcma.py

# DD-CMA-ESの概要

ここでは，以下のサンプルスクリプトを通して，DD-CMA-ESの概要と最適化結果（図）の解釈を説明します．


<!-- EN -->
# Overview of DD-CMA-ES


<!-- EN -->
Here, we explain the overview of DD-CMA-ES and how to interpret the optimization results (figures) through the following sample script.


## 実行スクリプトと実行結果

<!-- EN -->
## Execution Script and Results


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ellipsoid-Cigar function
N = 10

def ellcig(x):
    cig = np.ones(x.shape[1]) / np.sqrt(x.shape[1])
    d = np.logspace(0, 3, base=10, num=x.shape[1], endpoint=True)
    y = x * d
    f = 1e4 * np.sum(y ** 2, axis=1) + (1. - 1e4) * np.dot(y, cig)**2
    return f

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -5.0 * np.ones(N)
UPPER_BOUND = 5.0 * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return ellcig(xx)

# Setting for resart
NUM_RESTART = 10  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
xmean0 = np.random.randn(N)  # initial m
D0 = 2.0 * np.ones(N)        # initial D
ddcma = DdCma(xmean0=xmean0, sigma0=D0)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        xmean0 = np.random.randn(N)  # initial m
        D0 = 2.0 * np.ones(N)        # initial D
        ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

表示されている4列は左から，イテレーション数，目的関数評価回数，各イテレーションでのベスト解の目的関数値，これまでに得られたベスト解の目的関数値，を表しています．一番最後に，どの終了条件を満たして終了したかが表示されています．

<!-- EN -->
The four columns displayed are, from left to right: the iteration count, the number of objective function evaluations, the best objective function value among the candidate solutions generated in each iteration, and the best objective function value obtained so far. At the very end, the stopping criterion that triggered termination is displayed.


## 使用したテスト関数の紹介

<!-- EN -->
## Introduction to the Test Function Used



上のコードを実行した結果得られる最適化結果は，$d = 10$のEllipsoid-Cigar関数と呼ばれる関数
$$f(x) = 10^4 \sum_{i=1}^{d} \left(10^{3\frac{i-1}{d-1}} x_i\right)^2 + \frac{1 - 10^4}{d} \left(\sum_{i=1}^{d} 10^{3\frac{i-1}{d-1}} x_i\right)^2$$
を最適化した結果となります．この関数は凸二次関数ですが，変数毎に目的関数に与える影響が異なる（$10^{3\frac{i-1}{d-1}}$がかかっていることからわかります）ことに加え，$v = (1 / \sqrt{d}, \dots, 1 / \sqrt{d})$方向に対して目的関数が他の方向よりも鈍感である（第二項の影響です）といった特徴を持ちます．なお，この関数のヘッセ行列は
$$\nabla^2 f = 2 D_{\text{ell}} (10^4 I + (1 - 10^4) vv^T) D_{\text{ell}}$$
となります．ここで，$D_{\text{ell}} = \text{diag}(1, \dots, 10^{3\frac{i-1}{d-1}}, \dots, 10^{3})$です．また，その逆行列は
$$(\nabla^2 f)^{-1} = \frac{1}{2} D_{\text{ell}}^{-1} (10^{-4} I + vv^T) D_{\text{ell}}^{-1}$$
となります．


<!-- EN -->

The optimization results obtained by running the code above correspond to minimizing the Ellipsoid-Cigar function for $d = 10$,
$$f(x) = 10^4 \sum_{i=1}^{d} \left(10^{3\frac{i-1}{d-1}} x_i\right)^2 + \frac{1 - 10^4}{d} \left(\sum_{i=1}^{d} 10^{3\frac{i-1}{d-1}} x_i\right)^2$$
This function is a convex quadratic, but it has the characteristic that the influence on the objective function differs per variable (as indicated by the factor $10^{3\frac{i-1}{d-1}}$), and that the objective function is less sensitive in the direction $v = (1 / \sqrt{d}, \dots, 1 / \sqrt{d})$ than in other directions (due to the second term). The Hessian of this function is
$$\nabla^2 f = 2 D_{\text{ell}} (10^4 I + (1 - 10^4) vv^T) D_{\text{ell}}$$
where $D_{\text{ell}} = \text{diag}(1, \dots, 10^{3\frac{i-1}{d-1}}, \dots, 10^{3})$. Its inverse is
$$(\nabla^2 f)^{-1} = \frac{1}{2} D_{\text{ell}}^{-1} (10^{-4} I + vv^T) D_{\text{ell}}^{-1}$$


## DD-CMA-ESのアルゴリズムの紹介

DD-CMA-ESでは，多変量正規分布から複数の解を生成し，これらの目的関数を（多くの場合並列に）評価し，解のランキングを用いて多変量正規分布のパラメータを更新していきます．これを繰り返すことで，多変量正規分布を目的関数値の小さな（最小化を想定）領域へと収束させていきます．

DD-CMA-ESでは，多変量正規分布を$\mathcal{N}(m, \sigma^2 D C D)$ と表現します．ここで，$m \in \mathbb{R}^d$ は多変量正規分布の平均ベクトルを表し，プログラムやプロットにおいては xmean と書かれています．図のxmeanでは，$m$の各要素のイテレーション毎の変化がプロットされています．共分散行列は全体のスケーリングを表すステップサイズ $\sigma > 0$，要素毎のスケーリングを表す行列$D$（$d$次元対角行列），要素毎の相関を表す相関行列$C$の３つのパラメータを用いて表現されます．図のsigmaとDは$\sigma$と$D$の各要素を表しています．また，図中のSは相関行列$C$の$d$個の固有値の平方根の推移を表しています．なお，$\sigma D$がいわゆる分散行列であり，$D$自身も分布全体の大きさに関する情報を持っています．

今回の目的はアルゴリズムの解説ではないため，詳細は割愛します．[論文](https://direct.mit.edu/evco/article/28/3/405/94999/Diagonal-Acceleration-for-Covariance-Matrix#:~:text=https%3A//doi.org/10.1162/evco_a_00260)や[私が作成しているアルゴリズム解説用ノートブック](https://www.bbo.cs.tsukuba.ac.jp/research-j/cmaes%E9%80%B2%E5%8C%96%E6%88%A6%E7%95%A5%E3%81%AE%E8%A7%A3%E8%AA%AC)を参照してください．


<!-- EN -->
## Introduction to the DD-CMA-ES Algorithm


<!-- EN -->
DD-CMA-ES generates multiple candidate solutions from a multivariate normal distribution, evaluates their objective function values (in most cases in parallel), and updates the parameters of the multivariate normal distribution using the ranking of the solutions. By repeating this process, the multivariate normal distribution converges toward the region of small (assuming minimization) objective function values.

DD-CMA-ES represents the multivariate normal distribution as $\mathcal{N}(m, \sigma^2 D C D)$. Here, $m \in \mathbb{R}^d$ denotes the mean vector of the multivariate normal distribution, written as `xmean` in the program and plots. The xmean plot shows the iteration-by-iteration change of each element of $m$. The covariance matrix is represented using three parameters: the step size $\sigma > 0$ that represents overall scaling, the matrix $D$ (a $d$-dimensional diagonal matrix) that represents per-element scaling, and the correlation matrix $C$ that represents correlations between elements. The sigma and D plots in the figure show the values of $\sigma$ and each element of $D$, respectively. The S plot shows the evolution of the square roots of the $d$ eigenvalues of the correlation matrix $C$. Note that $\sigma D$ is what is commonly called the standard deviation matrix, and $D$ itself also carries information about the overall size of the distribution.

Since the goal here is not to explain the algorithm, we omit the details. Please refer to the [paper](https://direct.mit.edu/evco/article/28/3/405/94999/Diagonal-Acceleration-for-Covariance-Matrix#:~:text=https%3A//doi.org/10.1162/evco_a_00260) or [the algorithm-explanation notebook I have created](https://www.bbo.cs.tsukuba.ac.jp/research-j/cmaes%E9%80%B2%E5%8C%96%E6%88%A6%E7%95%A5%E3%81%AE%E8%A7%A3%E8%AA%AC).


## 結果の読み取り方

CMA-ESを理解し，活用していくうえで，上のような結果のグラフを解釈することが重要です．

複数の線がある xmean や D のプロットでは，赤系統がインデックスの小さな要素，青系統がインデックスの大きな要素に対応しています．S のプロットでは，固有値の平方根が昇順にソートされているので，カラーと要素のインデックスの間に関係はありません．

まず見るべきは，fmin です．これは，毎イテレーション生成されている解候補の中で，最小の目的関数値を表しています．最終的に望ましい解が得られているならば，実用上はそれ以上に議論する必要はないかもしれません．今回の問題では最適解の目的関数値は0であり，得られている解の目的関数値が指数的に減少している傾向が確認できます．

続いて，sigma を確認します．こちらも指数的に小さくなっていることがわかります．探索終了時点において，$\sigma^{(t)}/\sigma^{(0)} \approx 10^{-6}$程度になっていることから，初期の広がり$\sigma^{(0)}$と比較して十分に小さな分布になっていることがわかります．目的関数値が望ましい値なのかどうか以前に，正しく何らかの点に収束しているのかを見極めるためには，fmin でなく sigma を見ることが重要です．fmin が何らかの値で停滞しているように見える場合，収束しているのか，それとも探索が何らかの理由で進まなくなっているのかを fmin だけから見極めることは困難であるためです．例えば上の結果において，100〜200イテレーションの範囲において，目的関数値が停滞しているように見える箇所があります．目的関数値だけを見ていると，ここまでで収束したと見なしてしまうかもしれませんが，sigmaやSを見ると，共分散行列を適応している最中であることがわかります．

対角行列 D について注目すべきは，要素毎の比です．値が大きいということは，それだけその要素の方向に大きな標準偏差を多変量正規分布が持っていることになります．多変量正規分布がその方向に大きな広がりを持っているということは，目的関数が相対的にその方向への変化に対して鈍感であることを意味します．要素毎の目的関数値に与える感度が D によって学習されている様子が見て取れます．この結果から，この問題はインデックスの小さな変数が目的関数に与える影響は，インデックスの大きな変数が目的関数に与える影響よりも小さく，等高線を描いたとすれば各軸の長さの比が D の要素に比例した形になっているであろうことが予想されます．

最後に相関行列 C の固有値の平方根 S を確認します．イテレーション数が$2 \cdot 10^2$を超えたあたりから，S の最大の要素がその他の要素よりも$10^2$程度大きな値になっていることが見て取れます．これは，要素毎の感度をDが吸収したとしても，要素毎ではない何らかの１方向に対して目的関数に与える影響が鈍感な方向が存在していることがわかります．今回のテスト問題の場合，この方向は $v$ に対応しています．図からは読み取れませんが，以下のコードを実行すると，S の最大値に対応している固有ベクトルを確認することができます．

<!-- EN -->
## How to Interpret the Results


<!-- EN -->
Understanding and interpreting graphs of results like the ones above is important for understanding and utilizing CMA-ES.

In the xmean and D plots, which have multiple lines, reddish colors correspond to elements with smaller indices and bluish colors to elements with larger indices. In the S plot, the square roots of the eigenvalues are sorted in ascending order, so there is no correspondence between color and element index.

First, look at fmin. This represents the minimum objective function value among the candidate solutions generated in each iteration. If a satisfactory solution is ultimately obtained, there may be no need to discuss this further in practice. For the current problem, the optimal objective function value is 0, and it can be confirmed that the objective function value of the obtained solution decreases exponentially.

Next, check sigma. This can also be seen to decrease exponentially. At the end of the search, $\sigma^{(t)}/\sigma^{(0)} \approx 10^{-6}$, indicating that the distribution has become sufficiently small compared to the initial spread $\sigma^{(0)}$. To determine whether the optimization has converged to some point at all—before judging whether the objective function value is satisfactory—it is important to look at sigma rather than fmin. When fmin appears to plateau at some value, it is difficult to tell from fmin alone whether convergence has occurred or whether the search has stalled for some reason. For example, in the results above, there is a section between iterations 100 and 200 where the objective function value appears to plateau. Looking only at the objective function value might lead one to conclude convergence at that point, but looking at sigma and S reveals that the covariance matrix is still being adapted.

Regarding the diagonal matrix D, what deserves attention is the ratio between elements. A larger value means the multivariate normal distribution has a larger standard deviation in that element's direction. If the multivariate normal distribution has a large spread in a given direction, it means the objective function is relatively insensitive to changes in that direction. One can see how the per-element sensitivity to the objective function is learned by D. From these results, one can expect that this problem has smaller influence on the objective function from variables with smaller indices than from those with larger indices, and that if contours were drawn, the ratio of axis lengths would be proportional to the elements of D.

Finally, check S, the square roots of the eigenvalues of the correlation matrix C. Starting from around iteration $2 \cdot 10^2$, it can be seen that the largest element of S is about $10^2$ larger than the other elements. This indicates that even after D absorbs the per-element sensitivity, there exists some one direction (not per-element) in which the objective function is less sensitive. In the case of the current test problem, this direction corresponds to $v$. Although not visible in the figure, running the following code will allow you to check the eigenvector corresponding to the largest value in S.


In [ ]:
ddcma.B[:, -1]
# B は S の各要素に対応する C の単位固有ベクトル．
# S は昇順にソートされているので，最大値に対応する固有ベクトルは B の最後の列


経験的には，$D C D$は目的関数のヘッセ行列の逆行列$\nabla^2 f$に比例するような振る舞いが見られます．実際，今回の結果でも，$D$は$D_{\text{ell}}^{-1}$に概ね比例しており，$C$は$10^{-4} I + vv^T$に概ね比例していることが，上の実験結果からも確認されます．

<!-- EN -->

Empirically, $D C D$ tends to behave proportionally to the inverse of the Hessian of the objective function $\nabla^2 f$. Indeed, in the current results, it can be confirmed from the experimental results above that $D$ is roughly proportional to $D_{\text{ell}}^{-1}$ and $C$ is roughly proportional to $10^{-4} I + vv^T$.


#### まとめ：結果の図から分かること
- 十分に収束しているのか（sigma）
- 目的関数に対する感度が変数毎にどの程度異なるのか（D）
  - 感度が大きく異なる場合（例えばDの最大値と最小値の比が10以上），悪スケールな関数と呼ばれる
  - D の要素間の比が非常に大きい（例えば$10^6$以上）場合，数値的な誤差が大きくなる恐れがあるので，変数のスケールを調整しておく必要がある．
  - D の要素間の比が発散していくような場合，定式化時に注意が必要（後述）
- 変数間の依存関係が強いのか（S）
  - S の要素間の比が非常に大きい（例えば$10^6$以上）場合，数値的な誤差が大きくなる恐れがあるので，変数のスケールを調整しておく必要がある．
  - S の要素間の比が発散していくような場合，定式化時に注意が必要（後述）

目的関数値の推移だけを見て収束しているかどうかの判断をしてしまうのは危険です．必ず分布パラメータも見るようにしましょう．変数間依存性や悪スケール性のある問題（共分散行列を適応しなければ効率的に解けない問題）に対してCMA-ESは強力な方法ですが，共分散行列を適応している過程で目的関数値の減少が小さく見える場合が多くあります．

<!-- EN -->
#### Summary: What the result figures reveal
- Whether convergence is sufficient (sigma)
- How much the sensitivity to the objective function differs per variable (D)
  - When sensitivities differ greatly (e.g., the ratio of the maximum to minimum element of D exceeds 10), the function is called ill-scaled.
  - When the ratio between elements of D is very large (e.g., $10^6$ or more), numerical errors may become significant, so it is necessary to adjust the scale of the variables.
  - When the ratio between elements of D diverges, care is needed in the problem formulation (discussed later).
- Whether there are strong dependencies between variables (S)
  - When the ratio between elements of S is very large (e.g., $10^6$ or more), numerical errors may become significant, so it is necessary to adjust the scale of the variables.
  - When the ratio between elements of S diverges, care is needed in the problem formulation (discussed later).

It is dangerous to judge convergence solely from the history of objective function values. Always check the distribution parameters as well. CMA-ES is a powerful method for problems with inter-variable dependencies and ill-scaling (problems that cannot be solved efficiently without adapting the covariance matrix), but the decrease in objective function value often appears small during the process of adapting the covariance matrix.


# 最適化実行前に検討すべき項目

最適化実行前に検討しておくべきこととしては，以下の３点が挙げられます
1. 目的関数評価回数 & 次元数
2. 初期分布パラメータ
3. ハイパーパラメータ（集団サイズ$\lambda$）
3. リスタート & 終了条件

<!-- EN -->
# Items to Consider Before Running Optimization


<!-- EN -->
Items to consider before running optimization include the following three points:
1. Number of objective function evaluations & dimensionality
2. Initial distribution parameters
3. Hyperparameters (population size $\lambda$)
3. Restart & stopping criteria


## 1. 目的関数評価回数 & 次元数（CMA-ESを使うべきか）

まず，アルゴリズムの選択が不適切でないかを考えることが必要です．例えば，以下のような状況の場合には，CMA-ESよりも適切な方法があると考えられます．

- 次元数が高々５次元程度の場合：この場合，経験的にNelder-Mead法などの方が効率的であることが多く見られます．

- 許容できる実行時間内において，目的関数の評価回数が次元数の数十倍に満たない場合：この場合，CMA-ESは収束とみなせる程に分布が小さくなる前に終了してしまい，精度の高い解が得られない可能性が高いです．その場合，次元数が低い場合にはベイズ最適化のような最適化法を採用するか，さもなくばNelder-Mead法やその他の局所探索法，場合によってはベイズ最適化を採用したほうが現実的な時間内により望ましい解が得られる可能性が高いように思います．CMA-ESの場合，各イテレーションで生成される解候補は並列に評価することが可能なので，並列評価が可能なのであれば，これを考慮して最大評価回数を計算しましょう．

<!-- EN -->
## 1. Number of Objective Function Evaluations & Dimensionality (Should CMA-ES Be Used?)


<!-- EN -->
First, it is necessary to consider whether the choice of algorithm is appropriate. For example, in the following situations, a method more suitable than CMA-ES may exist.

- When the dimensionality is only about 5 or fewer: In this case, empirically, methods such as the Nelder-Mead method are often more efficient.

- When the number of objective function evaluations within the tolerable runtime is less than several tens of times the dimensionality: In this case, CMA-ES is likely to terminate before the distribution has become small enough to be considered converged, and there is a high probability that a high-accuracy solution will not be obtained. In that case, if the dimensionality is low, it is more realistic to use an optimization method such as Bayesian optimization, or otherwise the Nelder-Mead method, other local search methods, or Bayesian optimization as appropriate, as these may yield a more satisfactory solution within a practical amount of time. In CMA-ES, the candidate solutions generated in each iteration can be evaluated in parallel, so if parallel evaluation is possible, take this into account when calculating the maximum number of evaluations.


## 2. 初期分布パラメータ



初期分布のパラメータ $m^{(0)}$，$\sigma^{(0)}$, $D^{(0)}$, $C^{(0)}$ は，目的関数が複数の局所解を有する場合には重要な検討項目となります．ただし，予め変数毎のスケーリングなどを知らない限り$D^{(0)} = I$とし，変数間の依存関係などを知らない限り$C^{(0)} = I$とすることが自然です．以下，$m^{(0)}$と$\sigma^{(0)}$ について議論します．

- 有望な解の候補を知っている場合：例えば，既存の設計などの既存の解 $x_{\text{guess}}$ を持っている場合には，この解の周辺に初期化することが望ましいでしょう．これにより，少なくともこの初期解よりも優れた解の発見が期待できます．例えば，$m^{(0)} = x_{\text{guess}}$とし，$\sigma^{(0)}$を十分に小さな値とすれば，これを実現できます．ここで，$\sigma^{(0)}$は，対象としている問題毎に異なるでしょう．例えば，解の各要素が$10^{-4}$程度変化しただけではほとんど目的関数に影響しないであろう，などといった既存知識があるのであれば，$\sigma^{(0)} = 10^{-4}$などとすれば良いでしょう．変数毎にこの値が変わるようであれば，$\sigma^{(0)} = 1$とし，代わりに$D^{(0)}$の要素を変数毎に上と同様の方法で決定すれば良いでしょう．CMA-ESでは，分布の広がりが小さすぎる場合（すなわち大きく移動すれば目的関数を改善できるのに，広がりが狭すぎで移動できない場合），比較的高速に$\sigma$を大きくすることが可能なので，局所的な探索をしたい場合には，十分に小さな$\sigma^{(0)}$で初期化しておけば良いでしょう．

- 各設計変数の定義域が有限な場合 or 各設計変数の合理的な値の範囲を知っている場合：この場合，各設計変数の範囲$[L, U]$の中で，ランダムに初期化することが望ましいでしょう．例えば，$m^{(0)}_i \sim \mathcal{U}[L_i, U_i]$などと区間内の一様分布に従ってサンプリングし，$\sigma^{(0)} = 1$，$D^{(0)}_i = \frac{U_i - L_i}{4}$ などと初期化する方法が考えられます．ただし，$\sigma^{(0)}$については，より小さな値で初期化したほうがいい場合もあります．$\sigma^{(0)}$が大きいほど，特定の局所解に収束しがちになります．これが望ましい局所解であればよいのですが，そうでない場合，小さな$\sigma^{(0)}$としたほうが，多様な局所解を探索できる場合があります．

実際には，一度しか探索できないという場面は少ないでしょうから，有望な解の候補を知っている場合には，まずはその解を$m^{(0)}$，$\sigma^{(0)}$を適切に定めて探索し，その後$\sigma^{(0)}$を$10$倍して探索，$100$倍して探索，などと，徐々に大域的な探索を実行していくことが良いかと思います．なお，$\sigma^{(0)}$が大きくなるほど，$m^{(0)}$への依存性は下がっていきます．

<!-- EN -->
## 2. Initial Distribution Parameters


<!-- EN -->

The parameters of the initial distribution, $m^{(0)}$, $\sigma^{(0)}$, $D^{(0)}$, $C^{(0)}$, become important considerations when the objective function has multiple local optima. However, unless per-variable scaling information is known in advance, it is natural to set $D^{(0)} = I$, and unless inter-variable dependencies are known, to set $C^{(0)} = I$. The following discusses $m^{(0)}$ and $\sigma^{(0)}$.

- When a promising candidate solution is known: For example, if an existing solution $x_{\text{guess}}$ such as an existing design is available, it is desirable to initialize in the vicinity of this solution. This makes it possible to expect finding a solution at least better than this initial solution. For example, setting $m^{(0)} = x_{\text{guess}}$ and $\sigma^{(0)}$ to a sufficiently small value can achieve this. Here, $\sigma^{(0)}$ will differ for each target problem. For example, if there is prior knowledge such as "changing each element of the solution by about $10^{-4}$ would have almost no effect on the objective function," then setting $\sigma^{(0)} = 10^{-4}$ or similar would be appropriate. If this value varies per variable, set $\sigma^{(0)} = 1$ and instead determine the elements of $D^{(0)}$ per variable using the same method as above. In CMA-ES, when the distribution spread is too small (i.e., when a large move could improve the objective function but the spread is too narrow to move), it is possible to increase $\sigma$ relatively quickly, so when local exploration is desired, initializing with a sufficiently small $\sigma^{(0)}$ is appropriate.

- When the domain of each design variable is finite, or when the reasonable range of values for each design variable is known: In this case, it is desirable to initialize randomly within the range $[L, U]$ of each design variable. For example, one approach is to sample $m^{(0)}_i \sim \mathcal{U}[L_i, U_i]$ from a uniform distribution within the interval, and initialize $\sigma^{(0)} = 1$, $D^{(0)}_i = \frac{U_i - L_i}{4}$, etc. However, there are cases where initializing $\sigma^{(0)}$ to a smaller value is better. A larger $\sigma^{(0)}$ tends to converge to a specific local optimum. If that local optimum is the desired one, fine, but otherwise, a smaller $\sigma^{(0)}$ may allow exploration of a more diverse set of local optima.

In practice, since situations where you can only search once are rare, if a promising candidate solution is known, first search with $m^{(0)}$ set to that solution and $\sigma^{(0)}$ set appropriately, then search with $\sigma^{(0)}$ multiplied by 10, then by 100, and so on, gradually conducting more global searches. Note that as $\sigma^{(0)}$ becomes larger, the dependence on $m^{(0)}$ decreases.


### 2.1 10次元Rosenbrock 関数での例

Rosenbrock関数は
$$f(x) = \sum_{i=1}^{d-1} 100  \left(x_i^2 - x_{i+1}\right)^2 + \left( x_i - 1 \right)^2$$
と定義される四次関数です．最適解は$x^* = (1, \dots, 1)$となりますが，$(0, \dots, 0)$と$(1, \dots, 1)$をつなぐ曲線の周辺のみが低い目的関数値を取り，それ以外が相対的に高い目的関数値を取るような関数です．広い範囲を探索した場合，どこから探索してもまず原点付近に正規分布が一度集まるような傾向が見られます．

#### 確認事項
- 原点に平均ベクトルを初期化した場合，mode 1 ($\sigma^{(0)} = 1$)とmode 2 ($\sigma^{(0)} = 10^{-3}$)を比較することで，初期ステップサイズが小さすぎる場合には高速にこれを大きくできることを確認．
- 最適解周辺に平均ベクトルを初期化した場合，mode 3 ($\sigma^{(0)} = 10^{-1}$)とmode 4 ($\sigma^{(0)} = 1$)を比較することで，初期ステップサイズが十分に小さくないと局所探索にならない（良い解からスタートすることの意味があまりない）ことを確認．

#### 注意
初期ステップサイズを小さくしすぎた場合には，分布の発散を防ぐために標準で設けられている終了条件に引っかかってしまう恐れがある．その場合には初期ステップサイズが明らかに小さすぎるサインですので，10倍程度大きくして探索しましょう．

<!-- EN -->
### 2.1 Example with 10-Dimensional Rosenbrock Function


<!-- EN -->
The Rosenbrock function is defined as the quartic function
$$f(x) = \sum_{i=1}^{d-1} 100  \left(x_i^2 - x_{i+1}\right)^2 + \left( x_i - 1 \right)^2$$
The optimal solution is $x^* = (1, \dots, 1)$, but only the region near the curve connecting $(0, \dots, 0)$ and $(1, \dots, 1)$ takes low objective function values, while everywhere else takes relatively high values. When searching over a wide range, there is a tendency for the normal distribution to first converge near the origin regardless of where the search starts.

#### Items to verify
- When the mean vector is initialized at the origin, compare mode 1 ($\sigma^{(0)} = 1$) and mode 2 ($\sigma^{(0)} = 10^{-3}$) to confirm that when the initial step size is too small, it can be increased quickly.
- When the mean vector is initialized near the optimal solution, compare mode 3 ($\sigma^{(0)} = 10^{-1}$) and mode 4 ($\sigma^{(0)} = 1$) to confirm that if the initial step size is not small enough, the search does not become a local search (i.e., starting from a good solution has little meaning).

#### Note
If the initial step size is made too small, the standard stopping criterion implemented to prevent distribution divergence may be triggered. In that case, the initial step size is clearly too small, so increase it by a factor of about 10 and search again.


In [ ]:
# 実行スクリプト
# Rosenbrock function
N = 10

def rosenbrock(x):
    a = 1e2
    return a * np.sum(
        (x[:, :-1]**2 - x[:, 1:])**2, axis=1) + np.sum(
            (x[:, :-1] - 1.0)**2, axis=1)

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return rosenbrock(xx)

# Setting for resart
NUM_RESTART = 1   # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# 初期mと初期D （D0 の係数 1e-1 と 1e-4 を比較）
mode = 2
if mode == 0:
    # 原点に初期化
    xmean0 = np.zeros(N)
    D0 = 1e0 * np.ones(N)
elif mode == 1:
    # 原点に初期化（初期ステップサイズがかなり小さい場合）
    xmean0 = np.zeros(N)
    D0 = 1e-3 * np.ones(N)
elif mode == 2:
    # 最適解周りに初期化
    xmean0 = np.ones(N) + np.random.randn(N) * 1e-1
    D0 = 1e-1 * np.ones(N)
elif mode == 3:
    # 最適解周りに初期化（初期ステップサイズが大きい場合）
    xmean0 = np.ones(N) + np.random.randn(N) * 1e-1
    D0 = 1e0 * np.ones(N)

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        xmean0 = np.random.randn(N)  # initial m
        D0 = 2.0 * np.ones(N)        # initial D
        ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

### 2.1 10次元Rastrigin 関数での例

Rastrigin関数は
$$f(x) = \sum_{i=1}^{d} x_i^2 + 10 ( 1 - \cos(2 \pi x_i))$$
と定義される多峰性関数（局所解を複数持つ関数）です．最適解は$x^* = (0, \dots, 0)$となりますが，各座標の値が整数の値付近において，局所解が存在します．ただし，巨視的に見ると下凸のような景観をしています．

#### 確認事項
- $m^{(0)}$を$(1, \dots, 1)$付近に初期化した場合，初期ステップサイズが十分に小さければ（ここでは$\sigma^{(0)} = 0.1$），局所的に目的関数値が改善されること．
- ただし，あくまで局所的にしか改善されないため，比較的大きな$\sigma^{(0)}$のほうが良い局所解へと収束する場合があること．当然，悪くなる場合もあること．

<!-- EN -->
### 2.1 Example with 10-Dimensional Rastrigin Function


<!-- EN -->
The Rastrigin function is defined as the multimodal function (a function with multiple local optima)
$$f(x) = \sum_{i=1}^{d} x_i^2 + 10 ( 1 - \cos(2 \pi x_i))$$
The optimal solution is $x^* = (0, \dots, 0)$, but local optima exist near integer values of each coordinate. However, when viewed macroscopically, the landscape appears to be convex from below.

#### Items to verify
- When $m^{(0)}$ is initialized near $(1, \dots, 1)$, if the initial step size is sufficiently small (here $\sigma^{(0)} = 0.1$), the objective function value is improved locally.
- However, since improvement is only local, a relatively large $\sigma^{(0)}$ may lead to convergence to a better local optimum. Of course, there are cases where it gets worse.


In [ ]:
# 実行スクリプト
# Rosenbrock function
N = 10

def rastrigin(x):
    a = 1e1
    return np.sum(x[:, :]**2 + a * (1.0 - np.cos(2.0 * np.pi * x[:, :])), axis=1)

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return rastrigin(xx)

# Setting for resart
NUM_RESTART = 1  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# 初期mと初期D （D0 の係数 1e-1 と 1e-4 を比較）
mode = 0
if mode == 0:
    # ステップサイズが小さい場合
    xmean0 = np.ones(N) + np.random.randn(N) * 1e-1
    D0 = 1e-1 * np.ones(N)
elif mode == 1:
    # ステップサイズが相対的に大きい場合
    xmean0 = np.ones(N) + np.random.randn(N) * 1e-1
    D0 = 1.0 * np.ones(N)

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        xmean0 = np.random.randn(N)  # initial m
        D0 = 2.0 * np.ones(N)        # initial D
        ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

## 3. ハイパーパラメータ（集団サイズ$\lambda$）


CMA-ESのハイパーパラメータ全てに対して，最適化問題の次元数から求まる推奨値が設計されています．ただし，集団サイズ$\lambda$については，大きな値を与えることが望ましい場合があります．なお，$\lambda$を大きくすると，自動的に学習率などのパラメータの推奨値も変化します．

1. 目的関数の評価を，デフォルトの$\lambda$ ($= 4 + \lfloor 3 \log(d) \rfloor$)よりも多く並列計算可能である場合．一般に，$\lambda$を大きくすれば，探索に必要なイテレーション数は少なくなるため，探索終了までの実時間を削減することが可能です．ただし，$\lambda > d^2$程度になると，イテレーション数削減効果はほとんど見られなくなります．
2. 目的関数が多峰性関数である場合．集団サイズを大きくすることでより目的関数値の小さな局所解を得られる場合が多くあります．

<!-- EN -->
## 3. Hyperparameters (Population Size $\lambda$)


<!-- EN -->
For all hyperparameters of CMA-ES, recommended values derived from the dimensionality of the optimization problem are designed. However, for the population size $\lambda$, there are cases where providing a larger value is desirable. Note that increasing $\lambda$ also automatically changes the recommended values for parameters such as learning rates.

1. When it is possible to evaluate more objective functions in parallel than the default $\lambda$ ($= 4 + \lfloor 3 \log(d) \rfloor$): In general, increasing $\lambda$ reduces the number of iterations needed for optimization, which can reduce the real elapsed time until the end of the search. However, when $\lambda > d^2$ or so, the effect of reducing the number of iterations becomes negligible.
2. When the objective function is multimodal: Increasing the population size often allows obtaining local optima with smaller objective function values.


### 3.1 並列化のために集団サイズを増加させる効果の確認


10次元 Ellipsoid-Cigar 関数を用いて，デフォルトの集団サイズの場合（$\lambda = 10$，一番上の図）と大きな集団サイズ$\lambda = 20, 40, 80, \dots$の場合のイテレーション数を比較します．位置イテレーションに必要な評価を全て並列計算できる場合，実行時間はイテレーション数に概ね依存します．

<!-- EN -->
### 3.1 Checking the Effect of Increasing Population Size for Parallelization


<!-- EN -->

Using the 10-dimensional Ellipsoid-Cigar function, we compare the number of iterations for the default population size ($\lambda = 10$, top figure) and larger population sizes $\lambda = 20, 40, 80, \dots$. When all evaluations needed per iteration can be computed in parallel, the runtime depends approximately on the number of iterations.


In [ ]:
# 実行スクリプト
lam = 80

# Ellipsoid-Cigar function
N = 10

def ellcig(x):
    cig = np.ones(x.shape[1]) / np.sqrt(x.shape[1])
    d = np.logspace(0, 3, base=10, num=x.shape[1], endpoint=True)
    y = x * d
    f = 1e4 * np.sum(y ** 2, axis=1) + (1. - 1e4) * np.dot(y, cig)**2
    return f

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return ellcig(xx)

# Setting for resart
NUM_RESTART = 1   # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=np.random.randn(N), sigma0=np.ones(N)*2., lam=lam)  # 変更点はここ
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        ddcma = DdCma(xmean0=np.random.randn(N), sigma0=np.ones(N)*2., lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

集団サイズを大きくした場合に必要なイテレーション数が減少するのは，分布パラメータの更新に用いる学習率を大きく設定できる（推奨値もそのように設定されている）ためです．また，学習率の推奨値の設定から，$\lambda < 4d^2$程度まではイテレーション数の削減が確認できますが，それ以上ではイテレーション数は削減されません．

<!-- EN -->
When the population size is increased, the required number of iterations decreases because the learning rates used to update the distribution parameters can be set larger (and the recommended values are set accordingly). Also, from the recommended value settings for the learning rate, a reduction in the number of iterations can be confirmed up to about $\lambda < 4d^2$, but beyond that, the number of iterations is not reduced.


### 3.2 多峰性関数における集団サイズを増加させる効果の確認



上述のRastrigin関数のように，大域的に見れば下凸になっているようにみえる多峰性関数を大域的単峰な関数，もしくは大谷構造をもつ関数，などと呼びます．このような関数の場合，
- 初期ステップサイズを大きめに設定し，かつ，
- 集団サイズを大きめに設定

することで，目的関数値の低い局所解を獲得できる可能性があります．

#### 確認事項

- mode0とmode1を比較することで，集団サイズと初期ステップサイズがともに大きい場合には最適解発見確率が高くなるが，初期ステップサイズが小さすぎる場合には，集団サイズを大きくとっても局所探索になってしまうこと．(mode0でも必ず最適解が発見できるわけではないので，数回実行してみると良い)

<!-- EN -->
### 3.2 Checking the Effect of Increasing Population Size on Multimodal Functions


<!-- EN -->


Multimodal functions that appear convex from below when viewed globally, like the Rastrigin function mentioned above, are called globally unimodal functions, or functions with a global valley structure (ohya structure). For such functions,
- setting the initial step size larger, and
- setting the population size larger

may allow obtaining local optima with smaller objective function values.

#### Items to verify

- By comparing mode0 and mode1, confirm that when both the population size and the initial step size are large, the probability of finding the optimal solution is high, but when the initial step size is too small, the search becomes local even with a large population size. (mode0 does not always find the optimal solution, so try running it several times.)


In [ ]:
# 実行スクリプト
N = 10
xmean0 = np.ones(N) + np.random.randn(N) * 1e-1  # 局所解周辺に m を初期化
mode = 0
if mode == 0:
    # 集団サイズ大，初期ステップサイズ大
    lam = 200
    D0 = 2 * np.ones(N)
elif mode == 1:
    # 集団サイズ大，初期ステップサイズ小
    lam = 200
    D0 = 0.1 * np.ones(N)
elif mode == 2:
    # 集団サイズ小，初期ステップサイズ大
    lam = 10
    D0 = 2 * np.ones(N)
else:
    # 集団サイズ小，初期ステップサイズ小
    lam = 10
    D0 = 0.1 * np.ones(N)


def rastrigin(x):
    a = 1e1
    return np.sum(x[:, :]**2 + a * (1.0 - np.cos(2.0 * np.pi * x[:, :])), axis=1)

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return rastrigin(xx)

# Setting for resart
NUM_RESTART = 1  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=lam)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        ddcma = DdCma(xmean0=np.random.randn(N), sigma0=np.ones(N)*2., lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

一方で，例外も存在します．例えば，Double-Sphere関数（$s > 0$）
$$
f(x) = \min\left[ \sum_{i=1}^{d} (x_i - a_i)^2, d + s \sum_{i=1}^{d} (x_i - b_i)^2\right]
$$
がこれに該当します．この関数は二つの局所解しか持ちません．最適解は$a$で目的関数値が$f(a) = 0$，もう一つの局所解が$b$で目的関数値が$f(b) = d$となります．Rastrigin関数と異なり，大域的にみても二つの関数に分かれてしまっていることがわかります．また，$s$が小さい場合，局所解を形成する方の目的関数値のほうが小さい領域が広く存在しています．このような関数の場合，ステップサイズを大きくして大域的な探索を試みると，局所解の谷しか見えず（大域的最適解を形成する関数の領域に解が生成される確率が低い），局所解に収束する現象が見られます．このような関数を大域的多峰な関数であったり，UV構造を持つ関数などといいます．

#### 確認項目
Double Sphere関数の場合
- 初期ステップサイズを大きめに設定
- 集団サイズを大きめに設定

がいずれも逆効果になること

<!-- EN -->
On the other hand, exceptions also exist. For example, the Double-Sphere function ($s > 0$)
$$
f(x) = \min\left[ \sum_{i=1}^{d} (x_i - a_i)^2, d + s \sum_{i=1}^{d} (x_i - b_i)^2\right]
$$
falls into this category. This function has only two local optima. The global optimum is at $a$ with objective function value $f(a) = 0$, and the other local optimum is at $b$ with objective function value $f(b) = d$. Unlike the Rastrigin function, even when viewed globally, the function is divided into two parts. Moreover, when $s$ is small, there is a wider region where the objective function value forming the local optimum is smaller. For such a function, when attempting global search by increasing the step size, one sees only the valley of the local optimum (the probability that a solution is generated in the region of the function forming the global optimum is low), and a phenomenon of converging to the local optimum is observed. Such functions are called globally multimodal functions, or functions with a UV structure.

#### Items to verify
For the Double Sphere function, confirm that:
- setting a larger initial step size, and
- setting a larger population size

both have counterproductive effects.


In [ ]:
# 実行スクリプト
N = 10
xmean0 = np.zeros(N)
mode = 1
if mode == 0:
    # 集団サイズ大，初期ステップサイズ大
    lam = 200
    D0 = 5.0 * np.ones(N)
elif mode == 1:
    # 集団サイズ大，初期ステップサイズ小
    # 今回の問題はこれでも最適解を高い確率で発見できる
    lam = 200
    D0 = 1.0 * np.ones(N)
elif mode == 2:
    # 集団サイズ小，初期ステップサイズ大
    lam = 10
    D0 = 5.0 * np.ones(N)
else:
    # 集団サイズ小，初期ステップサイズ小
    lam = 10
    D0 = 1.0 * np.ones(N)


def doublesphere(x):
    s = 0.2
    a = 2.5
    b = - np.sqrt((a**2 - 1) / s)
    f1 = np.sum((x[:, :] - a)**2, axis=1)
    f2 = N + s * np.sum((x[:, :] - b)**2, axis=1)
    return np.fmin(f1, f2)

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = - np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return doublesphere(xx)

# Setting for resart
NUM_RESTART = 1  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=lam)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        ddcma = DdCma(xmean0=np.random.randn(N), sigma0=np.ones(N)*2., lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

## 4. リスタート & 終了条件



一度の探索で望ましい解が得られる可能性は現実的にはかなり低いでしょう．その場合，繰り返し探索を行うことが必須となります．その際，前項目で見たように，集団サイズや初期分布パラメータを変えながらリスタートすることが望ましいです．そのようなリスタート戦略は，IPOPメカニズム，BIPOPメカニズムなどとして，提案されています．今回のコードでは，IPOPメカニズム（リスタート毎に集団サイズを倍にする方法）が実装されています．

リスタート戦略を用いる際，利用者が検討すべき項目は，各リスタートの終了条件です．終了条件が適切でないと，探索が終わっていないのに打ち切ってしまう，もしくはいつまでもリスタートがかからない，といった問題があります．

上のコードでは，
- 分布が小さくなったら終了
- 共分散行列の条件数が大きくなりすぎたら終了
- 目的関数値が一定イテレーション以上改善されなければ終了
- ある目的関数値に到達したら終了（この場合，リスタートもかけない）
- 最大の目的関数評価回数に達したら終了（この場合，リスタートもかけない）

などの終了条件が実装されています．分布の大きさや条件数に関する終了条件のしきい値は，計算誤差を考慮して予め設定されていますが，実際に解いている問題によっては，ある程度以上変数が変化しない限りほとんど目的関数に差は無いから無視して良い，もしくはある程度以上の精度で解を社会実装できないから，それ以上の精度はそもそも必要ない，などといった事前情報がある場合があるでしょう．その場合には，しきい値を事前情報によって調整することで，効率的な探索が可能になる可能性があります．

<!-- EN -->
## 4. Restart & Stopping Criteria


<!-- EN -->

In practice, the probability of obtaining a satisfactory solution in a single search is quite low. In that case, it is essential to perform searches repeatedly. In doing so, as seen in the previous items, it is desirable to restart while varying the population size and initial distribution parameters. Such restart strategies have been proposed as the IPOP mechanism, the BIPOP mechanism, and others. The code in this tutorial implements the IPOP mechanism (a method that doubles the population size at each restart).

When using a restart strategy, the item the user needs to consider is the stopping criterion for each restart. If the stopping criterion is not appropriate, problems such as cutting off the search before it is finished, or never triggering a restart, may arise.

The code above implements stopping criteria such as:
- Stop when the distribution becomes small
- Stop when the condition number of the covariance matrix becomes too large
- Stop when the objective function value has not improved for a certain number of iterations or more
- Stop when a target objective function value is reached (in this case, no restart is triggered)
- Stop when the maximum number of objective function evaluations is reached (in this case, no restart is triggered)

The thresholds for stopping criteria related to the distribution size and condition number are preset considering computational errors. However, depending on the actual problem being solved, there may be prior information such as "differences in the objective function that arise unless the variable changes by a certain amount or more can be ignored," or "the solution cannot be implemented in practice with higher accuracy than a certain level, so accuracy beyond that is not needed in the first place." In such cases, by adjusting the thresholds based on prior information, it may be possible to achieve more efficient search.


In [ ]:
# 実行スクリプト
N = 10
xmean0 = np.ones(N) + np.random.randn(N) * 1e-1  # 局所解周辺
D0 = 0.1 * np.ones(N)

def rastrigin(x):
    a = 1e1
    return np.sum(x[:, :]**2 + a * (1.0 - np.cos(2.0 * np.pi * x[:, :])), axis=1)

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return rastrigin(xx)

# Setting for resart
NUM_RESTART = 10  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        xmean0 = -5.0 + 10.0 * np.random.rand(N)  # リスタート時の初期分布パラメータ
        D0 = 2.5 * np.ones(N)                     # リスタート時の初期分布パラメータ
        ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

# 定式化の見直し



最適化を実行してみたが望ましい解が得られない，という場面は多々あると思います．まず前項目までにみた検討項目について，十分な回数のリスタートがなされているのか，適切な終了条件が設定されているのか，初期分布パラメータは適切か，など，再度確認してみましょう．それでもうまく行かない場合，最適化問題の定式化を再検討することが必要になります．その際，分布パラメータの推移（上で出力しているような図）も確認しましょう．その中に，うまく解くためのヒントが隠されている場合があります．

ここでは，実際に起こりがちな，CMA-ESにとって解きにくい目的関数の設計と，そのときのCMA-ESの振る舞いをいくつか紹介します．

1. ほとんど平らな多峰性の目的関数 or 周期関数
2. 目的関数に影響を与えない変数
3. 尖った等高線を持つ目的関数
4. 大域的多峰な目的関数

<!-- EN -->
# Revisiting the Problem Formulation


<!-- EN -->

There are many cases where a satisfactory solution cannot be obtained after running optimization. First, re-examine the items considered in the previous sections: whether a sufficient number of restarts have been performed, whether appropriate stopping criteria are set, whether the initial distribution parameters are appropriate, and so forth. If things still do not work well, it becomes necessary to reconsider the formulation of the optimization problem. At that time, also check the history of the distribution parameters (figures such as those output above). Hints for solving the problem well may be hidden there.

Here, we introduce several practically common objective function designs that are difficult for CMA-ES to solve, and the behavior of CMA-ES in those cases.

1. Nearly flat multimodal objective functions or periodic functions
2. Variables that have no effect on the objective function
3. Objective functions with sharp contours
4. Globally multimodal objective functions


## 1. ほとんど平らな多峰性の目的関数 or 周期関数


CMA-ESは目的関数の値を直接用いず，複数の解のランキングに基づいて探索をしていきます．そのため，どれだけ勾配が小さくても，解の優越が正しく決定できる限り，勾配の小さな問題でも全く問題なく最適化ができます．例えば，
$$
f(x) = \sum_{i=1}^{d} x_i^2
$$
と
$$
f(x) = 20 - 20 \exp\left(-0.2 \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2}\right)
$$
は，単調増加変換の関係にあるため，CMA-ESからは等価な関数であるように扱われます．これは，目的関数のスケーリングを考慮しなくて良い，という観点で望ましい性質の一つです．

ただし，ここに多峰性が含まれると，注意すべき状況が発生します．例えば，上の関数に周期関数を加えた Ackley関数
$$
f(x) = 20 - 20 \exp\left(-0.2 \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2}\right) + \exp(1) - \exp\left(\frac{1}{d}\sum_{i=1}^{d} \cos(2\pi x_i)\right)
$$
を考えましょう．局所解は複数存在しますが，探索領域が$[-30, 30]^d$程度に制限されている場合，この関数はCMA-ESにとって，最適解$x^* = 0$を比較的容易に発見できる関数です．しかし，その探索領域が制限されていない場合，$[-30, 30]^d$の外側では追加された項$\exp(1) - \exp\left(\frac{1}{d}\sum_{i=1}^{d} \cos(2\pi x_i)\right)$が支配的になります．この場合，CMA-ESからみれば，ほとんど周期関数を複数の周期に渡って最適化しているように見えます．周期関数を複数周期に渡って探索しようとした場合，挙動が不安定になる場合があります．

#### 確認事項
以下，Ackley関数を，と，探索領域を設けて最適化した場合の比較です．
- 探索領域を設けずに最適化した場合（初期ステップサイズを30と比較的大きめに設定），$\sigma  D$ の値が以上に大きな値となり，収束できないか非常に効率が悪いこと
- 制約条件を設けることで，不安定な振る舞いを回避できること



<!-- EN -->
## 1. Nearly Flat Multimodal Objective Functions or Periodic Functions


<!-- EN -->

CMA-ES does not use objective function values directly; it searches based on the ranking of multiple solutions. Therefore, no matter how small the gradient is, as long as the dominance between solutions can be correctly determined, optimization can be performed without any problem even for problems with small gradients. For example,
$$
f(x) = \sum_{i=1}^{d} x_i^2
$$
and
$$
f(x) = 20 - 20 \exp\left(-0.2 \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2}\right)
$$
are related by a monotonically increasing transformation, so they are treated as equivalent functions by CMA-ES. This is one of the desirable properties in the sense that the scaling of the objective function does not need to be considered.

However, when multimodality is included here, a situation requiring care arises. For example, consider the Ackley function, which adds a periodic function to the above function:
$$
f(x) = 20 - 20 \exp\left(-0.2 \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2}\right) + \exp(1) - \exp\left(\frac{1}{d}\sum_{i=1}^{d} \cos(2\pi x_i)\right)
$$
Multiple local optima exist, but when the search domain is restricted to approximately $[-30, 30]^d$, this function is one for which CMA-ES can find the optimal solution $x^* = 0$ relatively easily. However, when the search domain is not restricted, outside of $[-30, 30]^d$, the added term $\exp(1) - \exp\left(\frac{1}{d}\sum_{i=1}^{d} \cos(2\pi x_i)\right)$ becomes dominant. In that case, from CMA-ES's perspective, it appears almost as if it is optimizing a periodic function over multiple periods. When attempting to search a periodic function over multiple periods, the behavior may become unstable.

#### Items to verify
The following compares optimization of the Ackley function with and without a search domain constraint.
- When optimizing without a search domain constraint (initial step size set to 30, relatively large), the values of $\sigma D$ become abnormally large, and convergence cannot be achieved or is very inefficient.
- Adding constraints avoids the unstable behavior.


In [ ]:
# 実行スクリプト
# Ellipsoid-Cigar function
N = 10
xmean0 = 30 * np.random.rand(N)  # 恣意的に最適解からずらして初期化
D0 = 30 * np.ones(N)  # 探索領域の幅の1/4
def ackley(x):
    a = 20
    b = 0.2
    c = 2 * np.pi
    f1 = a * (1 - np.exp(-b*np.sqrt(np.mean(x**2, axis=1))))
    f2 = np.exp(1) - np.exp(np.mean(np.cos(c * x), axis=1))
    return f1 + f2

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
if False:
    # 制約条件を考慮しない場合
    LOWER_BOUND = -np.inf * np.ones(N)
    UPPER_BOUND = np.inf * np.ones(N)
else:
    # 制約条件を考慮する場合
    LOWER_BOUND = -30.0 * np.ones(N)
    UPPER_BOUND = 30.0 * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return ackley(xx)

# Setting for resart
NUM_RESTART = 10  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        xmean0 = 30 * np.random.rand(N)  # 恣意的に最適解からずらして初期化
        D0 = 15 * np.ones(N)  # 探索領域の幅の1/4
        ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

#### まとめ
- 最適解から遠く離れた領域の目的関数値が一定値に近づいていくようなケースでは，CMA-ESの振る舞いが不安定になり，探索が非効率になるか収束しないケースがあります．そのような目的関数を回避するか，合理的な設計変数の範囲が分かる場合には制約を設けましょう．
- $\sigma D$ の値が以上に大きくなっている現象が見られた場合，この問題が起こっている可能性が高いので，制約を設けるなどの工夫を試みてみると良いでしょう．

#### 補足（制約について）
制約を設けた場合に xmean が制約の外側に収束していないように見えるのは，ミラーリングという制約対処を用いており，制約の外側に仮想的な景観を作り出して探索しているためです．得られた xmean をミラーリングすると制約の内側の解が得られていることがわかります．

<!-- EN -->
#### Summary
- In cases where the objective function value approaches a constant value far from the optimal solution, CMA-ES behavior may become unstable, making the search inefficient or failing to converge. Avoid such objective functions, or if the reasonable range of design variables is known, impose constraints.
- If a phenomenon is observed where the values of $\sigma D$ become abnormally large, this problem is likely occurring, so try measures such as imposing constraints.

#### Supplementary note (on constraints)
The reason xmean does not appear to converge outside the constraint boundary when constraints are imposed is that a technique called mirroring is used for constraint handling, which creates a virtual landscape outside the constraint boundary for exploration. Mirroring the obtained xmean reveals that a feasible solution inside the constraint boundary is obtained.


In [ ]:
print(ddcma.xmean)
print(mirror(ddcma.xmean, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC))

## 2. 目的関数に影響を与えない変数



目的関数に影響を与えない変数が多く含まれる場合，それらを取り除いた場合と比較して探索効率が著しく悪化する現象が見られます．
このような冗長な変数は，例えばover-parameterizedな回帰モデルの係数を最適化する場合など，しばしば現れます．

例えば，$d = 20$次元のSphere関数
$$
f(x) = \sum_{i=1}^{d} x_i^2
$$
と$d = 100$次元であるがそのうち$20$個の変数しか意味を持たないsubspace-Sphere関数
$$
f(x) = \sum_{i=1}^{\lfloor d / 5 \rfloor} x_i^2
$$
で挙動を比較してみましょう．

#### 確認事項
- 本質的には同じ目的関数であるが，冗長な変数が存在すると$\sigma$の減少速度が遅くなり，収束までにより多くの評価回数を費やすこと
- 冗長な変数が存在する場合，共分散行列の条件数が発散していくこと（その結果，数値誤差が大きくなる恐れがある）

<!-- EN -->
## 2. Variables That Have No Effect on the Objective Function


<!-- EN -->

When many variables that have no effect on the objective function are included, a phenomenon of significantly reduced search efficiency compared to the case where those variables are removed is observed.
Such redundant variables arise frequently, for example when optimizing the coefficients of an over-parameterized regression model.

For example, let us compare the behavior between the $d = 20$-dimensional Sphere function
$$
f(x) = \sum_{i=1}^{d} x_i^2
$$
and the $d = 100$-dimensional subspace-Sphere function where only $20$ out of the $d$ variables are meaningful:
$$
f(x) = \sum_{i=1}^{\lfloor d / 5 \rfloor} x_i^2
$$

#### Items to verify
- Although essentially the same objective function, the presence of redundant variables slows down the rate of $\sigma$ decrease, causing more evaluations to be spent until convergence.
- When redundant variables are present, the condition number of the covariance matrix diverges (which may result in larger numerical errors).


In [ ]:
# 実行スクリプト
mode = 0
if mode == 0:
    # 冗長な設計変数
    N = 100
    def subspace_sphere(x):
        dd = 50
        return np.sum(x[:, :dd]**2, axis=1)
elif mode == 1:
    # 冗長な設計変数を取り除いた場合
    N = 50
    def sphere(x):
        return np.sum(x[:, :]**2, axis=1)

xmean0 = 5 * np.random.rand(N)
D0 = 5 * np.ones(N)

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    if mode == 0:
        return subspace_sphere(xx)
    else:
        return sphere(xx)

# Setting for resart
NUM_RESTART = 1  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        xmean0 = 5 * np.random.rand(N)
        D0 = 5 * np.ones(N)
        ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

#### まとめ
- 冗長な変数がある場合，
  - $\sigma$ の減少速度の低下による探索効率の低下
  - 共分散行列の条件数の発散
- 前者の問題は，冗長な変数の割合が少なければほとんど影響は現れないため，現実的な時間で最適化が進んでいれば特に気にする必要はないかもしれません
- 後者の問題は，求めたい設計変数の精度が高く精緻化が必要な場合には，条件数が発散することによる数値誤差の問題が発生し得ます．そのような場合，探索結果として得られた共分散行列を用いて，次元削減することをおすすめします．一例を以下にあげます．

<!-- EN -->
#### Summary
- When redundant variables are present:
  - Reduced search efficiency due to a slower rate of $\sigma$ decrease
  - Divergence of the condition number of the covariance matrix
- The former problem has almost no effect when the proportion of redundant variables is small, so it may not need to be a concern if optimization is progressing within a practical amount of time.
- The latter problem may cause numerical error issues due to the divergence of the condition number when high precision of the desired design variables is needed. In such a case, it is recommended to perform dimensionality reduction using the covariance matrix obtained as the search result. An example is given below.


In [ ]:
Cov = ddcma.transform(ddcma.transform(np.eye(ddcma.N)).T)  # 共分散行列を作成
eigval, eigvec = np.linalg.eigh(Cov)  # 固有値分解，eigvec[:, i] が eigval[i] に対応する単位固有ベクトル
print('共分散行列の固有値', eigval)  # 固有値の大きさが大きいもの（冗長な次元）と小さいもの（必要な次元）に分かれていることを確認

N = np.sum(eigval < 1e-8)  # 次元削減後の次元数
xbase = np.copy(ddcma.xmean)  # 前回探索時の平均ベクトルを座標系の原点とする
basis = np.copy(eigvec[:, :N])  # 次元削減後の基底

def reconstruct(x):
    # 次元削減後の（CMA-ESが新たに探索する）設計変数からもとの変数を復元
    return xbase + np.dot(x, basis.T)

In [ ]:
# 実行スクリプト
xmean0 = np.zeros(N)  # 原点が前回探索した解なので，その周辺から始めることがおすすめ
D0 = np.ones(N)

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return subspace_sphere(reconstruct(xx))  # CMA-ESが探索する空間は次元削減後の空間なので，目的関数にわたすときには reconstruct

# Setting for resart
NUM_RESTART = 1  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        xmean0 = 5 * np.random.rand(N)
        D0 = 5 * np.ones(N)
        ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')


## 3. 尖った等高線を持つ目的関数



CMA-ESは勾配を利用せず，また目的関数の値も直接は使用しないため，微分可能かどうかが直接は探索のしやすさに影響しません．例えば，なめらかな関数を単調増加変換した関数はなめらかになる保証はなく，勾配も定義されない可能性がありませんが，CMA-ESにとってはなめらかな関数と同等に扱われます．なめらかな関数の単調変換で表されるような関数は，等高線がなめらかになります．

一方，等高線が尖った目的関数の場合には，注意が必要です．
例えば，
$$
f(x) = \sum_{i=1}^{d} |x_i|
$$
という関数は，$x_i = 0$となる任意の点において，等高線が尖っていることが容易にわかります．これを一般化したものとして，
$$
f(x) = \left(\sum_{i=1}^{d} |x_i|^{p}\right)^{1/p}
$$
なども考えられます．$p$が小さい値になると，より尖った等高線となります．

等高線が尖っていても，その角度が0でなければ（例えば$p = 1$のケース），うまく探索できるでしょう．しかし，その角度が0になるケースでは，失敗する場合があります．

#### 確認事項
- $p = 1/4$のとき，デフォルトの集団サイズでは最適解でない微分不能な点に早期収束してしまう
- 集団サイズを増加させることで緩和することは可能

<!-- EN -->
## 3. Objective Functions with Sharp Contours


<!-- EN -->

Since CMA-ES does not use gradients and does not directly use objective function values, whether a function is differentiable does not directly affect the ease of search. For example, a function obtained by applying a monotonically increasing transformation to a smooth function is not guaranteed to be smooth and may not have a well-defined gradient, but CMA-ES treats it equivalently to a smooth function. Functions expressible as monotonic transformations of smooth functions have smooth contours.

On the other hand, care is needed for objective functions with sharp contours.
For example, the function
$$
f(x) = \sum_{i=1}^{d} |x_i|
$$
can easily be seen to have sharp contours at any point where $x_i = 0$. As a generalization of this,
$$
f(x) = \left(\sum_{i=1}^{d} |x_i|^{p}\right)^{1/p}
$$
can also be considered. When $p$ is small, the contours become sharper.

Even if contours are sharp, if the angle is not zero (e.g., the $p = 1$ case), the search can proceed well. However, when the angle becomes zero, failure may occur.

#### Items to verify
- When $p = 1/4$, the default population size causes premature convergence to a non-optimal point of non-differentiability.
- Increasing the population size can mitigate this.


In [ ]:
# 実行スクリプト
N = 10

def pnorm(x):
    p = 1/4  # ここを変更して実行
    f = np.sum(np.abs(x)**p, axis=1)**(1/p)
    return f

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = -5.0 * np.ones(N)
UPPER_BOUND = 5.0 * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return pnorm(xx)

# Setting for resart
NUM_RESTART = 10  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=np.random.randn(N), sigma0=np.ones(N)*2.)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        ddcma = DdCma(xmean0=np.random.randn(N), sigma0=np.ones(N)*2., lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

#### まとめと対策
- 尖った等高線を持つ目的関数では探索に失敗することがある．特に，開き角度が$0$になるケースは注意．
- 多少の尖ったケースであれば，集団サイズを大きくすることで，対応可能です．集団サイズを大きくしながらリスタートする戦略を取っていれば，問題なく対応できる場合があります．ただし，定式化の段階で対応するに越したことはありません．
- 尖った等高線を持つ目的関数は，max や min，条件分岐がある場合に起こりやすい．可能であれば，max をなめらかな関数で近似するなどの工夫をしましょう．例えば，maxであればLogSumExp関数（$a > 0$）
$$
z_{\max} = \max(z_1, \dots, z_k) \approx \frac{1}{a} \log\left(\sum_{i=1}^{k} \exp(a z_i)\right) = z_{\max} + \frac{1}{a} \log\left(\sum_{i=1}^{k} \exp(a (z_i - z_\max))\right)
$$
であったり，Softmax関数を用いる方法が考えられます．
- 制約条件をペナルティ関数を用いて対処する場合も，尖った等高線を作り出しやすいので，注意が必要です．例えば，ある関数$g(x)$が$G$を超えてほしくない場合，$f(x) + c \max(g(x) - G, 0)$（$c$はペナルティ係数）といった方法がよく採用されますが，この場合も制約の境界上がなめらかでなくなる可能性があります．境界上の目的関数値が最適でない場合には，問題になりませんが，最適であるため境界上に収束することが必要になる場合には気をつけましょう．



<!-- EN -->
#### Summary and countermeasures
- Objective functions with sharp contours may cause search failure. In particular, cases where the opening angle becomes $0$ require attention.
- For moderately sharp cases, increasing the population size can handle the problem. If you use a strategy of restarting while increasing the population size, you may be able to handle the problem without issues. However, it is even better to address the issue at the formulation stage.
- Objective functions with sharp contours tend to arise when there are max, min, or conditional branches. If possible, try approximating max with a smooth function. For example, for max, the LogSumExp function ($a > 0$)
$$
z_{\max} = \max(z_1, \dots, z_k) \approx \frac{1}{a} \log\left(\sum_{i=1}^{k} \exp(a z_i)\right) = z_{\max} + \frac{1}{a} \log\left(\sum_{i=1}^{k} \exp(a (z_i - z_\max))\right)
$$
or the Softmax function can be used.
- When dealing with constraints using penalty functions, sharp contours are also easily created, so care is needed. For example, when a function $g(x)$ should not exceed $G$, a method such as $f(x) + c \max(g(x) - G, 0)$ (where $c$ is a penalty coefficient) is often adopted, but in this case too, the boundary of the constraint may become non-smooth. If the optimal objective function value is not at the boundary, this is not a problem, but care must be taken when it is necessary to converge to the boundary because the optimal solution lies there.


## 4. 大域的多峰な目的関数



前述の通り，大域的多峰性を持つ関数の場合，最適化アルゴリズムサイドからアプローチしようとしても，集団サイズを大きくしても効果が無い，もしくは逆効果，となるため，初期値を変えながらリスタートを繰り返す力技に頼らざるを得ません．

この場合，目的関数の設計を工夫することで，回避できる場合もあります．例えば，目的関数とは別にある関数$g(x)$が存在し，望ましい解であれば$g(x)$の値も小さくなるはずである，という事前知識がある場合，これが活用できます．例えば，$f(x) + c g(x)$（$c > 0$）などとし，$c$をうまく調整することで，大域的最適な解へと誘導することができる場合もあります．

例えば，前述のdouble-Sphere関数を考えましょう．最適解の座標は各次元$a > 0$，もう一つの局所解の座標は各次元$b < 0$ となっていました．例えば事前知識として，$g(x) = \sum_{i=1}^{d}(x_i - 1)^2$が小さい解が望ましいといった情報を持っているとしましょう．このとき，上のように目的関数を変更することで，$g(x)$の値が小さくなるような解へと誘導することができます．ただし，そのようにして得られた解は，必ずしも$f(x)$の（局所）最適解ではないため，得られた解を初期解として，$f(x)$を局所探索すると良いでしょう．

#### 確認事項
Double-Sphere 関数を用いて
- 適切な関数$g$を用いることで，大域的最適解の谷へと誘導できること
- 得られた解を用いて局所探索することで，$f$の最適解が得られること

<!-- EN -->
## 4. Globally Multimodal Objective Functions


<!-- EN -->


As mentioned above, for functions with global multimodality, even from the algorithm side, increasing the population size has no effect or is counterproductive, so one has no choice but to rely on the brute-force approach of repeatedly restarting with different initial values.

In some cases, this can be circumvented by devising the objective function design. For example, if there exists a function $g(x)$ separate from the objective function, and there is prior knowledge that if a solution is desirable then $g(x)$ should also be small, this can be exploited. For example, by forming $f(x) + c g(x)$ ($c > 0$) and adjusting $c$ appropriately, it may be possible to guide the search toward the globally optimal solution.

For example, consider the Double-Sphere function mentioned earlier. The coordinates of the optimal solution were $a > 0$ in each dimension, and the coordinates of the other local optimum were $b < 0$ in each dimension. Suppose, for example, that prior knowledge indicates that solutions with small values of $g(x) = \sum_{i=1}^{d}(x_i - 1)^2$ are desirable. By modifying the objective function as above, one can guide the search toward solutions with smaller values of $g(x)$. However, since the solution obtained in this way is not necessarily a (local) optimal solution of $f(x)$, it is advisable to use the obtained solution as the initial solution for a local search of $f(x)$.

#### Items to verify
Using the Double-Sphere function:
- Confirm that using an appropriate function $g$ can guide the search toward the valley of the global optimum.
- Confirm that by performing local search using the obtained solution, the optimal solution of $f$ can be obtained.


In [ ]:
# 実行スクリプト
N = 10
xmean0 = np.zeros(N)
D0 = 5.0 * np.ones(N)
lam = 10

def doublesphere(x):
    s = 0.2
    a = 2.5
    b = - np.sqrt((a**2 - 1) / s)
    f1 = np.sum((x[:, :] - a)**2, axis=1)
    f2 = N + s * np.sum((x[:, :] - b)**2, axis=1)
    return np.fmin(f1, f2)

def g(x):
    return np.mean((x - 1.0)**2, axis=1)

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = - np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return doublesphere(xx) + 1e1 * g(xx)  # ガイドを追加

# Setting for resart
NUM_RESTART = 1  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=lam)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        ddcma = DdCma(xmean0=np.random.randn(N), sigma0=np.ones(N)*2., lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

In [ ]:
xmean0 = np.copy(ddcma.xmean)
print(xmean0, doublesphere(xmean0.reshape((1, -1))))

In [ ]:
# 実行スクリプト
N = 10
D0 = 0.1 * np.ones(N)  # 上のようにして最適化して得られた平均ベクトルを初期解として局所探索
lam = 10

# Support for box constraint and periodic variables
# Set np.nan, -np.inf or np.inf if no bound
LOWER_BOUND = - np.inf * np.ones(N)
UPPER_BOUND = np.inf * np.ones(N)
FLAG_PERIODIC = np.asarray([False] * N)
period_length = (UPPER_BOUND - LOWER_BOUND) * 2.0
period_length[FLAG_PERIODIC] /= 2.0
period_length[np.logical_not(np.isfinite(period_length))] = np.inf

def fobj(x):
    xx = mirror(x, LOWER_BOUND, UPPER_BOUND, FLAG_PERIODIC)
    return doublesphere(xx)

# Setting for resart
NUM_RESTART = 1  # number of restarts with increased population size
MAX_NEVAL = 1e6   # maximal number of f-calls
F_TARGET = 1e-8   # target function value
total_neval = 0   # total number of f-calls

# Main loop
ddcma = DdCma(xmean0=xmean0, sigma0=D0, lam=lam)
ddcma.upper_bounding_coordinate_std(period_length)
checker = Checker(ddcma)
logger = Logger(ddcma)
for restart in range(NUM_RESTART):
    issatisfied = False
    fbestsofar = np.inf
    while not issatisfied:
        ddcma.onestep(func=fobj)
        ddcma.upper_bounding_coordinate_std(period_length)
        fbest = np.min(ddcma.arf)
        fbestsofar = min(fbest, fbestsofar)
        if fbest <= F_TARGET:
            issatisfied, condition = True, 'ftarget'
        else:
            issatisfied, condition = checker()
        if ddcma.t % 10 == 0:
            print(ddcma.t, ddcma.neval, fbest, fbestsofar)
            logger()
    logger(condition)
    print("Terminated with condition: " + str(condition))
    # For restart
    total_neval += ddcma.neval
    if total_neval < MAX_NEVAL and fbest > F_TARGET and restart + 1 < NUM_RESTART:
        popsize = ddcma.lam * 2
        ddcma = DdCma(xmean0=np.random.randn(N), sigma0=np.ones(N)*2., lam=popsize)
        checker = Checker(ddcma)
        logger.setcma(ddcma)
        print("Restart with popsize: " + str(ddcma.lam))
    else:
        break

# Produce a figure
fig, axdict = logger.plot()
for key in axdict:
    if key not in ('xmean'):
        axdict[key].set_yscale('log')
plt.tight_layout()
plt.savefig(logger.prefix + '.pdf')

## その他のトピック


<!-- EN -->
## Other Topics



#### 制約条件の対処
制約と一言でいっても，使える情報や計算時間など，状況は多岐に渡ります．矩形制約などは標準的に対応されている場合が多いものの，それ以外の制約はペナルティ関数を使うなどが一般的に利用しやすいでしょう．ただし，ペナルティ係数の設定によっては非常に解きにくい問題になってしまう，最適解が制約を違反してしまう，などの困難さがつきまといます．制約条件の性質毎に適切な制約対処法が異なるため，性質を良く検討した上で制約対処法を選択しましょう．

#### シミュレーション条件の不確実性
目的関数は基本的にシミュレーションを通して計算されると想定していますが，シミュレーションの条件が予め一意に定まるとは限りません．例えば，タイヤの設計をしている場合，路面状態によってグリップ性能は変わるでしょう．その場合，特定の状況を想定して最適化しても，他の状況では良くない解を得てしまう可能性があります．また，現実環境の情報が不足しているため，シミュレーション条件がそもそも不確実である，という場面も想定されます．その場合，考え得る最悪ケースの性能を最適化する，などの方法があります．シミュレーション環境が不確実な場合には，そのようなロバスト最適化などを検討しましょう．

#### 実行時間
目的関数の評価回数（シミュレーション回数）が少ない場合，CMA-ESよりもNelder-Mead法などを用いた局所探索をしたほうが，良い解を得られる場合があることを述べました．しかし，目的関数の評価回数が少なければ，あくまで局所的な探索しかできないことも事実です．大域的な探索を行いたい場合，シミュレーションの精度を落として高速に計算できるようにする方法が考えられます．ただ，愚直にこれを実装すると，シミュレーション精度が悪いので探索結果の信頼性が損なわれます．これに対して，シミュレーション精度をコントロールしながら最適化を進めていく，マルチフィデリティ最適化，という方針があります．シミュレーション時間がネックでありCMA-ESを活用できない場合には，この方針を検討してみましょう．

<!-- EN -->

#### Constraint handling
Even when we use the word "constraint," the available information and computation time vary widely. Box constraints are often handled as standard, but for other types of constraints, using penalty functions is generally the most accessible approach. However, difficulties arise such as the problem becoming very hard to solve depending on the penalty coefficient setting, or the optimal solution violating the constraint. Since the appropriate constraint handling method differs depending on the nature of the constraints, carefully examine the properties and select an appropriate constraint handling method.

#### Uncertainty in simulation conditions
It is assumed that the objective function is basically computed through simulation, but the simulation conditions are not always uniquely determined in advance. For example, when designing a tire, grip performance varies depending on road surface conditions. In that case, even if optimization is performed assuming a specific situation, there is a possibility of obtaining a poor solution in other situations. Also, situations can be imagined where simulation conditions are inherently uncertain due to insufficient information about the real environment. In that case, there are methods such as optimizing the worst-case performance. When the simulation environment is uncertain, consider such robust optimization.

#### Runtime
We noted that when the number of objective function evaluations (number of simulations) is small, local search using the Nelder-Mead method may yield a better solution than CMA-ES. However, it is also true that when the number of objective function evaluations is small, only local search is possible. When global search is desired, a method of reducing simulation accuracy to enable faster computation can be considered. However, implementing this naively undermines the reliability of the search results due to poor simulation accuracy. In response to this, there is an approach called multi-fidelity optimization, which proceeds with optimization while controlling simulation accuracy. When simulation time is a bottleneck and CMA-ES cannot be utilized effectively, consider this approach.


# おわりに

<!-- EN -->
# Conclusion


この資料がCMA-ESの利用者にとって少しでもCMA-ESをうまく活用する糧になればと思いますが，実応用上の困難さは多岐に渡るため，カバーしきれていないトピックも多くあります．どうしてもうまく行かないという場合には，秋本までご相談ください．

<!-- EN -->
I hope this material will be of some help to CMA-ES users in making better use of CMA-ES, but since the difficulties in practical applications are diverse, there are many topics not covered. If you find that things simply do not work out, please feel free to consult Akimoto.
